# Breakout 4B · Sprint 2 · four questions, four lists

From the declaration lines (imports and import VAT paid, under the tax number used to clear) and the VAT returns. A trader goes on the list of the first question it fails: 1 not known, 2 not filing, 3 import not in its books, 4 sells below what it imports, 5 consistent for now. **I01 is worked.**

*Published customs datasets record an irregularity on 1 to 8 lines in 100; in this lab the random draw finds 3 in 100.*

In [ ]:
# Run this cell first (click it, then Shift + Enter). It finds the lab data.
import hashlib, os
from pathlib import Path
import numpy as np, pandas as pd

CANDIDATES = [os.environ.get('WORKSHOP_DATA', ''), 'data', '../data', '/workspace/data', '.']
DATA = next(Path(c) for c in CANDIDATES if c and (Path(c) / 'tariff.csv').exists())
print('Data folder:', DATA)

def _h(v, nd):
    return hashlib.sha256(f'{round(float(v), nd):.{nd}f}'.encode()).hexdigest()[:16]
def _hs(v):
    return hashlib.sha256(str(v).strip().lower().encode()).hexdigest()[:16]
def check(name, value, expected, nd=None):
    if callable(value):
        try:
            value = value()
        except Exception:
            value = ...
    try:
        ok = value is not ... and (_hs(value) == expected if nd is None else _h(value, nd) == expected)
    except Exception:
        ok = False
    shown = value if nd is None or value is ... else f'{value:,.{nd}f}'
    print(('✓ ' if ok else '✗ not yet: ') + name + ' = ' + str(shown))


In [ ]:
imp = pd.read_csv(DATA / 'importers_4b.csv')
pairs = pd.read_csv(DATA / 'pairs_4b.csv')
returns = pd.read_csv(DATA / 'vat_returns.csv')
usual = pd.read_csv(DATA / 'sector_usual_4b.csv')   # computed for you: a median per sector
cols = ['period_month', 'importer_id', 'tin_customs', 'regime', 'customs_value_lcu', 'vat_lcu']
lines = pd.concat([pd.read_csv(DATA / f, usecols=cols, low_memory=False) for f in ('declarations_h1.csv', 'declarations_h2.csv')])
vat_lines = lines[(lines.period_month >= 25) & lines.regime.isin(['home_use', 'warehouse_out'])]   # the lines that bear VAT, last 12 months
periods = sorted(returns.period.unique()); returns['pno'] = returns.period.map({p: i + 1 for i, p in enumerate(periods)})
# your sprint 1 classes: the tax number on the tax side
conf = pairs[['ref', 'exact_status', 'best_tin', 'confidence']]
imp = imp.merge(conf, on='ref').merge(usual[['sector', 'sector_usual']], on='sector')
imp['tin_tax'] = np.where(imp.exact_status == 'matched', imp.tin_customs, np.where(imp.confidence >= 0.9, imp.best_tin, None))

## Worked: I01 through the four questions

In [ ]:
r = imp.set_index('ref').loc['I01']
imports = vat_lines[vat_lines.tin_customs == r.tin_customs].customs_value_lcu.sum()
paid = vat_lines[vat_lines.tin_customs == r.tin_customs].vat_lcu.sum()
g = returns[returns.tin == r.tin_tax]
filed6 = ((g.pno >= 19) & (g.filed == 1)).sum()
deducted = g[g.pno >= 13].import_vat_credit_lcu.sum(); sales = g[g.pno >= 13].sales_lcu.sum()
print('known: yes ·', 'filed', filed6, 'of 6 ·', f'booked {deducted / paid:.0%} ·', f'sales ÷ imports {sales / imports:.2f}')
print('list 4 · VAT at stake', round(0.10 * (imports * r.sector_usual - sales) / 1e6, 1), 'M LCU')

## YOUR TURN: the 25

In [ ]:
# YOUR TURN: for every importer, the first question failed (1 to 5) and the VAT at stake
imp['imports'] = ...   # with no number on the declaration: its own lines with a blank number
imp['paid'] = ...
imp['filed6'] = ...
imp['deducted'] = ...
imp['sales'] = ...
imp['question'] = ...  # 1 no tax number on the tax side; 2 filed < 5; 3 deducted ÷ paid < 0.5; 4 sales ÷ imports < 1; else 5
imp['stake'] = ...     # lists 1-2: 0.10 × imports × (usual − 1); lists 3-4: 0.10 × (imports × usual − sales); never below 0

In [ ]:
E = {'n1': 'ef2d127de37b942b', 'n2': 'ef2d127de37b942b', 'n3': 'd4735e3a265e16ee', 'n4': '4b227777d4dd1fc6', 'n5': '19581e27de7ced00', 's1': '1dcba2891fbb1d2f', 's2': 'c45ba7a8031c130d', 's3': '1304fd19c5b91656', 's4': '98beadf571b15167'}
for k in range(1, 6):
    check(f'importers failing question {k}' if k < 5 else 'consistent for now', lambda: (imp.question == k).sum(), E[f'n{k}'], 0)
for k in range(1, 5):
    check(f'VAT at stake, list {k} (M LCU)', lambda: imp[imp.question == k].stake.sum() / 1e6, E[f's{k}'], 1)